<div align="right"><a href="S34-Burg-Algorithm.ipynb">Sprache wechseln zu Englisch 🇬🇧</a></div>

Die folgenden Übungen sind Teil des Buches
* K.-D. Kammeyer, K. Kroschel: "Digitale Signalverarbeitung - Filterung und Spektralanalyse", Springer Vieweg, 2022.  [[KK22]](https://doi.org/10.1007/978-3-663-09805-8)

---

Dieses Jupyter Notebook kann durch Klicken auf das Symbol rechts in [Google Colab](https://colab.research.google.com/notebooks/intro.ipynb) geöffnet werden. Am besten wird es jedoch lokal mit [Jupyter Notebook](https://jupyter.org/) angezeigt (z. B. als Teil der [Anaconda](https://www.anaconda.com/)-Distribution). Beachte bitte, dass zur Ausführung von MATLAB in Notebooks zunächst die [MATLAB API für Python](https://mathworks.com/help/matlab/matlab-engine-for-python.html) installiert werden muss. Die Installation wird [hier](https://mathworks.com/help/matlab/matlab_external/install-the-matlab-engine-for-python.html) beschrieben. <a href="https://colab.research.google.com/github/Stefan-Goe/dsp-book/blob/main/matlab/L34-Burg-Algorithmus.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open Notebook in Google Colab"></a>

---

<!--NAVIGATION-->
<div align="center">

Zurück: [L33 Yule-Walker-/Autokorrelationsmethode](./L33-Yule-Walker-und-Autokorrelationsmethode.ipynb) | [Aufgabe 34](./A34-Burg-Algorithmus.ipynb) | [Inhalt / Übersicht](./00-MATLAB-Notebooks-Overview-DE.ipynb) | Weiter: [L35 Vergleich von Spektralschätzverfahren](./L35-SpektraleSchaetzverfahren.ipynb)

</div>

---


## Aufgabe: Burg

Die Grundidee des Burg-Algorithmus besteht darin, das Gleichgewicht der Leistungen  
des Vorwärts- und Rückwärtsprädiktionsfehlers auszunutzen  
und beide Leistungen gemeinsam zu minimieren.  
Dieser Ansatz liefert eine unverzerrte Schätzung der Koeffizienten des AR($n$)-Modells  
(im Gegensatz zur Autokorrelationsmethode); zusätzlich ist der Analysefilter  
immer ein Minimumphasenfilter (im Gegensatz zur Autokovarianzmethode).

Wir beginnen mit einem (Allpol-)System  

$H(z) = \frac{1}{A(z)}$  

mit  

$$
A(z) = 1 + 0.3611z^{-1} - 0.1373z^{-2} + 0.2685z^{-3}
- 0.0533z^{-4} - 0.3353z^{-5} - 0.307z^{-6}
$$

der Ordnung $m = 6$, welches durch einen stationären, mittelwertfreien, weißen  
Rauschprozess $N(k)$ mit der Leistung $\sigma_N^2 = 1$ angeregt wird  
(Befehl: `filter(·)`).

<img src="./pngs/l15sys.eps" style="width:70%;">

---

### a)

Plotten Sie die Impulsantwort $h(k)$ des Systems im Intervall  

$$
0 \le k \le 80
$$

unter Verwendung des Befehls `impz(·)`  
und stellen Sie das Pol-Nullstellen-Diagramm mit dem Befehl `zplane(·)` dar.

---

### b)

Schätzen Sie die Koeffizienten eines AR-Systems AR($n$)

$$
\frac{1}{A_n(z)} = \frac{1}{a_0 + a_1 z^{-1} + \cdots + a_n z^{-n}}
$$

auf Basis von $N = 2^{13}$ Stichproben des Prozesses $X(k)$  
für verschiedene Ordnungen $n = 2,\; 6,\; 32$  
(Befehl: `lburg(·)`).

Vergleichen Sie die unterschiedlich geschätzten Leistungsdichtespektren  
untereinander sowie mit dem exakten Leistungsdichtespektrum  
$S_{XX}(e^{j\Omega})$.

Warum ist die Schätzung für $n = 6$ besonders gut?

---

### c)

Wiederholen Sie die Aufgaben a) und b) für ein FIR-System

$$
H(z) = 1 - 4z^{-1} + 6z^{-2} - 4z^{-3} + z^{-4}
$$

der Ordnung $m = 4$.

Beobachten Sie die Anordnung der Pole des AR($n$)-Modells  
zur Kompensation der (vierfachen) Nullstelle von $H(z)$ bei $z = 1$.

In [ ]:
# BEGIN SOLUTION
MA = [1]; 
AR = [1.0000 0.3611 -0.1373 0.2685 -0.0533 -0.3353 -0.307];
NFFT=2^10;

% #####  Teilaufgabe a:  AR-System, Einschwingverhalten #####
% exakte AKF (eingeschw. Zustand)
[h_arma,t] = impz(MA,AR,80);

% # grafische Ausgabe # %
figure;
stem(t,h_arma); 
grid; 
xlabel('k'); 
ylabel('h(k)');
title(sprintf('Impulsantwort des AR(%d)-Modells',length(AR)-1));

figure; 
[h1,h2,h3]=zplane(MA,AR); set(h1,'Visible','off');
title(sprintf('Pole des AR(%d)-Modells',length(AR)-1)); 
grid;
xlabel('Realteil'); 
ylabel('Imaginaerteil');

input('....Zum Fortfahren: RETURN druecken');

% #####  Teilaufgabe b:  AR-System, LDS #####
% Analyse der Filterordnung AR(n)
Omega_norm = 0:1/NFFT:1-1/NFFT;
Sxx_arma = abs(fft(MA,NFFT)./fft(AR,NFFT)).^2;
n = [2 length(AR)-1 32];
N = 2^13;
N_vor = 80;
Noise = randn(1,N+N_vor);
x_all = filter(MA,AR,Noise);
x = x_all(N_vor+1:length(x_all));

for k = 1:3
    ar = lburg_algo(x,N,n(k),MA,AR);
    Sxx_ar = abs(ones(1,NFFT)./fft(ar,NFFT)).^2;
    Sxx_ar = Sxx_ar/sum(Sxx_ar).*sum(Sxx_arma);
    % # grafische Ausgabe # %
    figure;
    h1 = plot(Omega_norm, 10*log10(Sxx_arma/max(Sxx_arma)),'r--');  
    hold on;
    h2 = plot(Omega_norm, 10*log10(Sxx_ar/max(Sxx_ar)),'b-');
    ylabel('S_{xx}(e^{j*\Omega}) in dB');
    title(sprintf('LDS von Burg-AR(%d), N=%d', n(k),N)); 
    xlabel('\Omega/2\pi'); 
    grid;
    % legend([h1(1),h2(1)],'exaktes LDS','geschaetztes LDS',3);
    
    figure; 
    zplane(AR,ar); 
    title(sprintf('Pole von Burg-AR(%d)',n(k)));
    xlabel('Realteil'); 
    ylabel('Imaginaerteil'); 
    grid;
end;

input('....Zum Fortfahren: RETURN druecken');

% #####  Teilaufgabe c:  MA-System #####
% Analyse der Filterordnung AR(n)
MA = [1 -4 6 -4 1]; 
AR = 1;

% Einschwingverhalten
[h_arma,t] = impz(MA,AR);

% # grafische Ausgabe # %
figure; 
stem(t,h_arma); 
grid; 
xlabel('k'); 
ylabel('h(k)');
title(sprintf('Impulsantwort des MA(%d)-Modells',length(MA)-1));

figure; 
zplane(MA,AR);
title(sprintf('Pole des MA(%d)-Modells',length(MA)-1));
xlabel('Realteil'); 
ylabel('Imaginaerteil'); 
grid;

input('....Zum Fortfahren: RETURN druecken');

% Analyse der Filterordnung AR(n), LDS
Omega_norm  = 0:1/NFFT:1-1/NFFT;
Sxx_arma = abs(fft(MA,NFFT)./fft(AR,NFFT)).^2;

n = [2 6 32];
N = 2^13;
N_vor = 5;
noise = randn(1,N+N_vor);
x_all = filter(MA,AR,noise);
x = x_all(N_vor+1:length(x_all));

for k = 1:3
    ar = lburg_algo(x,N,n(k),MA,AR);
    Sxx_ar = abs(ones(1,NFFT)./fft(ar,NFFT)).^2;
    Sxx_ar = Sxx_ar/sum(Sxx_ar).*sum(Sxx_arma);
    
    % # grafische Ausgabe # %
    figure;
    Sxx_ar(find(Sxx_ar==0)) = 1e-10;
    Sxx_arma(find(Sxx_arma==0)) = 1e-10;
    h1 = plot(Omega_norm, 10*log10(Sxx_arma/max(Sxx_arma)),'r--');  
    hold on;
    h2 = plot(Omega_norm, 10*log10(Sxx_ar/max(Sxx_ar)),'b-');
    xlabel('\Omega/2\pi'); 
    ylabel('S_{xx}(e^{j*\Omega}) in dB');
    title(sprintf('LDS von Burg-AR(%d), N=%d', n(k),N)); 
    grid;
    % legend([h1(1),h2(1)],'exaktes LDS','geschaetztes LDS',3);
    axis([0 1 -70 0]);
    
    figure; 
    zplane(AR,ar); 
    title(sprintf('Pole von Burg-AR(%d)',n(k)));
    xlabel('Realteil'); 
    ylabel('Imaginaerteil'); 
    grid;
end;
% #####  EOF  #####
# END SOLUTION

## Lösung: Burg

### a)

Die folgende Abbildung zeigt die Impulsantwort $h(k)$ des AR(6)-Systems  
im Intervall $0 \le k \le 80$  
sowie das zugehörige Pol-Nullstellen-Diagramm  
mit den Polen

- $z_{\infty,1} = 0.3444 + i\cdot 0.8315$  
- $z_{\infty,2} = z_{\infty,1}^{*}$  
- $z_{\infty,3} = 0.85$  
- $z_{\infty,4} = -0.91$  
- $z_{\infty,5} = -0.49 + i\cdot 0.4950$  
- $z_{\infty,6} = z_{\infty,5}^{*}$  

<div style="display:flex; justify-content:center; gap:20px;">
  <img src="./pngs/ldsv9a03_a1.png" style="width:45%;">
  <img src="./pngs/ldsv9a03_a2.png" style="width:45%;">
</div>

---

### b)

Verfahren zur Spektralschätzung (nach Yule–Walker bzw. Burg)  
versuchen, einen AR-gefilterten weißen Rauschprozess zu modellieren  
(d. h. die AR-Parameter  
$\hat{a}_{0,n}, \ldots, \hat{a}_{n,n}$ zu bestimmen),  
sodass dessen statistische Eigenschaften (insbesondere die ACF)  
möglichst gut denen des gegebenen  
(stationären Prozesses mit beliebigem Spektrum) $X(k)$ entsprechen.

Dies funktioniert besonders gut, wenn der Prozess tatsächlich auf  
einer AR-Filterung beruht und wenn die Ordnung des AR($n$)-Modells  
der Ordnung des verwendeten Filters entspricht.  
Der Fall einer Filterung mit einem AR-System der Ordnung $m = 6$  
ist nachfolgend dargestellt.

<div style="display:flex; justify-content:center; gap:20px;">
  <img src="./pngs/ldsv9a03_b11.png" style="width:45%;">
  <img src="./pngs/ldsv9a03_b12.png" style="width:45%;">
</div>

Die beiden oberen Abbildungen zeigen das exakte Leistungsdichtespektrum  
im Vergleich zu den Leistungsdichtespektren, deren AR-Koeffizienten  
mit dem Burg-Algorithmus für die Ordnungen  
$n = 2, 6, 32$ geschätzt wurden.

Die unteren Abbildungen zeigen die zugehörigen Polpositionen,  
wobei die Pole von $H(z)$ ausnahmsweise als *Kreise* dargestellt sind.  
Alle Berechnungen basieren auf einer einzelnen Stichprobenfolge  
der Länge $N = 2^{13} = 8192$, um eine statistisch hinreichende  
Schätzung zu gewährleisten.

Ist die Ordnung des AR-Modells unterschätzt (z. B. $n = 2$),  
versucht der Burg-Algorithmus, einen möglichst guten Kompromiss  
bei der Platzierung der Pole zu finden.  
Dennoch besteht eine deutliche Abweichung zwischen dem geschätzten  
und dem exakten Leistungsdichtespektrum.

<div style="display:flex; justify-content:center; gap:20px;">
  <img src="./pngs/ldsv9a03_b21.png" style="width:45%;">
  <img src="./pngs/ldsv9a03_b22.png" style="width:45%;">
</div>

Entspricht die Ordnung des AR($n$)-Modells der Ordnung des verwendeten  
Filters $H(z)$, so ist die Approximationsgüte – wie erwartet – am besten.

<div style="display:flex; justify-content:center; gap:20px;">
  <img src="./pngs/ldsv9a03_b31.png" style="width:45%;">
  <img src="./pngs/ldsv9a03_b32.png" style="width:45%;">
</div>

Im Fall einer Überabschätzung der Modellordnung mit $n = 32$  
werden alle verbleibenden Koeffizienten nahezu null gesetzt:

$$
\hat{a}_{n,i} \approx 0 \quad \text{für} \quad i > m
$$

Diese Nullen entsprechen $n$ Polen, die auf einem Kreis nahe dem  
Einheitskreis liegen. Dadurch weist das resultierende  
Leistungsdichtespektrum einen stärker „welligen“ Verlauf auf.  
Die Verschlechterung der Approximation bleibt jedoch gering.

---

### c)

Die folgende Abbildung zeigt die Impulsantwort $h(k)$ des MA(4)-Systems  
im Intervall $0 \le k \le 4$.

$H(z)$ ist ein MA-Modell der Ordnung $m = 4$, sodass die Impulsantwort  
eine endliche Länge von $m + 1 = 5$ besitzt:

$$
h(k) = \{1, -4, 6, -4, 1\}
$$

Damit ist die Energiekorrelationsfunktion  
$r_{HH}^{(E)}(\kappa)$ und folglich auch  
$r_{XX}(\kappa)$  
auf den Bereich $|\kappa| \le 4$ beschränkt.

Es ergeben sich die Werte:

- $r_{XX}(0) = 70$  
- $r_{XX}(\pm 1) = -56$  
- $r_{XX}(\pm 2) = 28$  
- $r_{XX}(\pm 3) = -8$  
- $r_{XX}(\pm 4) = 1$  
- $r_{XX}(\kappa) = 0$ für $|\kappa| \ge 5$

Beachten Sie, dass $H(z)$ eine vierfache Nullstelle bei $z = 1$ besitzt  
(Hochpassverhalten).

<div style="display:flex; justify-content:center; gap:20px;">
  <img src="./pngs/ldsv9a03_c1.png" style="width:45%;">
  <img src="./pngs/ldsv9a03_c2.png" style="width:45%;">
</div>

---

### d)

Mit dem Burg-Algorithmus wird nun das MA-Modell $H(z)$  
durch AR-Modelle $1/A_n(z)$ unterschiedlicher Ordnung  
$n = 2, 6, 32$ approximiert.

<div style="display:flex; justify-content:center; gap:20px;">
  <img src="./pngs/ldsv9a03_d11.png" style="width:45%;">
  <img src="./pngs/ldsv9a03_d12.png" style="width:45%;">
</div>

<div style="display:flex; justify-content:center; gap:20px;">
  <img src="./pngs/ldsv9a03_d21.png" style="width:45%;">
  <img src="./pngs/ldsv9a03_d22.png" style="width:45%;">
</div>

<div style="display:flex; justify-content:center; gap:20px;">
  <img src="./pngs/ldsv9a03_d31.png" style="width:45%;">
  <img src="./pngs/ldsv9a03_d32.png" style="width:45%;">
</div>

Die dargestellten Ergebnisse zeigen, dass der Burg-Algorithmus  
die Nullstellen durch eine Konstellation von Polen nahe dem  
Einheitskreis approximiert.

Bei niedriger Ordnung ($n = 2$) werden die Pole möglichst weit  
von der Nullstelle entfernt platziert und auf einem Kreis  
in Richtung der Nullstelle angeordnet.  
Der unmittelbare Bereich um die Nullstelle bleibt dabei frei.

Anhand der geschätzten Spektraldichten ist erkennbar,  
dass mit steigender Ordnung des AR-Modells  
eine zufriedenstellende Approximation erreicht wird.